라이브러리 임포트

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import os
import warnings
import polars as pl
import requests
import pandas as pd
import csv
import io
import time
from pathlib import Path

In [103]:
# ==================================================
# 1. 설정
# ==================================================

start_month = '2008-01'

# 지난달까지 다운로드
end_month = (
    pd.Timestamp.now(tz='Asia/Seoul')
    .tz_localize(None)
    .to_period('M') - 1
).strftime('%Y-%m')

# 특정 기간만 받고 싶으면 변경
# end_month = '2008-12'

save_dir = Path(
    '/Users/bestw/Desktop/ex/python_ex/PythonProject/'
    '개인학습/git_local_folder/공공데이터/인구분석/인구데이터'
)
save_dir.mkdir(parents=True, exist_ok=True)

url = 'https://jumin.mois.go.kr/downloadCsvAge.do'

month_list = pd.period_range(
    start=start_month,
    end=end_month,
    freq='M'
)

failed_months = []
download_count = 0
skip_count = 0


# ==================================================
# 2. CSV 검증
# ==================================================

def validate_csv(content, year, month):
    # 실제 다운로드 파일은 CP949
    text = content.decode('cp949')

    reader = csv.reader(io.StringIO(text))
    columns = next(reader, [])

    prefix = f'{year}년{month}월'

    required_columns = {
        '행정구역',
        f'{prefix}_계_0세',
        f'{prefix}_남_0세',
        f'{prefix}_여_0세',
        f'{prefix}_남_99세',
        f'{prefix}_여_99세',
    }

    if not required_columns.issubset(columns):
        raise ValueError(
            'CSV 컬럼 또는 조회 월이 다릅니다. '
            '오류 페이지가 반환되었을 수도 있습니다.'
        )

    # 행정구역 1개 + (인구합계 2개 + 연령 101개) × 계·남·여
    if len(columns) != 310:
        raise ValueError(
            f'예상 컬럼 수는 310개인데 {len(columns)}개입니다.'
        )

    # 다른 월의 데이터가 섞였는지도 확인
    if any(not col.startswith(prefix + '_') for col in columns[1:]):
        raise ValueError('요청한 월과 다른 월의 컬럼이 있습니다.')

    row_count = 0

    for row in reader:
        if not row:
            continue

        if len(row) != len(columns):
            raise ValueError(
                f'{row_count + 1}번째 데이터 행의 컬럼 수가 다릅니다.'
            )

        row_count += 1

    # 전국 읍면동 자료인지 확인하기 위한 최소 건수 점검
    # 이 기준만으로 전국 자료의 완전성을 보장하는 것은 아님
    if row_count < 3000:
        raise ValueError(
            f'데이터가 {row_count:,}행입니다. '
            '전국 전체 읍면동 자료인지 확인이 필요합니다.'
        )

    return row_count


# ==================================================
# 3. 월별 다운로드
# ==================================================

with requests.Session() as session:
    session.headers.update({
        'User-Agent': 'Mozilla/5.0',
        'Referer': 'https://jumin.mois.go.kr/ageStatMonth.do'
    })

    for idx, period in enumerate(month_list, start=1):
        year = str(period.year)
        month = f'{period.month:02d}'
        year_month = f'{year}{month}'

        file_name = (
            f'{year_month}_{year_month}_연령별인구현황_월간.csv'
        )

        file_path = save_dir / file_name
        temp_path = file_path.with_suffix('.csv.part')

        label = f'[{idx}/{len(month_list)}] {year}-{month}'

        # 기존 파일도 내용을 검사한 후 건너뛰기
        if file_path.exists():
            try:
                content = file_path.read_bytes()
                row_count = validate_csv(content, year, month)

                print(
                    f'{label} | 기존 파일 정상 → 건너뛰기 | '
                    f'{len(content) / 1_000_000:.2f}MB | '
                    f'{row_count:,}행'
                )

                skip_count += 1
                continue

            except (ValueError, OSError) as e:
                print(f'{label} | 기존 파일 검증 실패: {e}')
                print('다시 다운로드합니다.')

        # 화면에서 전체읍면동현황을 선택했을 때의 요청 조건
        params = {
            'searchYearMonth': 'month',
            'xlsStats': '3'
        }

        data = {
            'sltOrgType': '1',
            'sltOrgLvl1': 'A',       # 전국
            'sltOrgLvl2': '',
            'gender': 'gender',      # 남·여 포함
            'sum': 'sum',            # 계 포함
            'sltUndefType': '',      # 등록구분: 전체
            'searchYearStart': year,
            'searchMonthStart': month,
            'searchYearEnd': year,
            'searchMonthEnd': month,
            'sltOrderType': '1',
            'sltOrderValue': 'ASC',
            'sltArgTypes': '1',      # 1세 단위
            'sltArgTypeA': '0',
            'sltArgTypeB': '100',
            'category': 'month',
            'state': '3'             # 전체 읍면동
        }

        # 일시적인 오류는 최대 3회 시도
        for attempt in range(1, 4):
            try:
                print(
                    f'{label} | 다운로드 중 '
                    f'({attempt}/3회)',
                    flush=True
                )

                response = session.post(
                    url,
                    params=params,
                    data=data,
                    timeout=(15, 180)
                )
                response.raise_for_status()

                content = response.content
                row_count = validate_csv(content, year, month)

                # 검증을 통과한 데이터만 최종 파일로 저장
                temp_path.write_bytes(content)
                temp_path.replace(file_path)

                download_count += 1

                print(
                    f'{label} | 저장 완료 | '
                    f'{len(content) / 1_000_000:.2f}MB | '
                    f'{row_count:,}행'
                )
                break

            except (requests.RequestException, ValueError, OSError) as e:
                print(f'{label} | 실패: {e}')

                if attempt < 3:
                    time.sleep(5 * attempt)
                else:
                    failed_months.append(year_month)

        # 월별 요청 간격
        time.sleep(2)


# ==================================================
# 4. 최종 결과
# ==================================================

print('\n다운로드 작업 완료')
print(f'조회 기간: {start_month} ~ {end_month}')
print(f'신규 다운로드: {download_count:,}개')
print(f'기존 정상 파일: {skip_count:,}개')
print(f'실패한 월: {len(failed_months):,}개')

if failed_months:
    print('실패 목록:', ', '.join(failed_months))

print(f'저장 위치: {save_dir}')

[1/225] 2008-01 | 기존 파일 정상 → 건너뛰기 | 6.47MB | 3,888행
[2/225] 2008-02 | 기존 파일 정상 → 건너뛰기 | 6.45MB | 3,881행
[3/225] 2008-03 | 기존 파일 정상 → 건너뛰기 | 6.46MB | 3,881행
[4/225] 2008-04 | 기존 파일 정상 → 건너뛰기 | 6.46MB | 3,881행
[5/225] 2008-05 | 기존 파일 정상 → 건너뛰기 | 6.44MB | 3,872행
[6/225] 2008-06 | 기존 파일 정상 → 건너뛰기 | 6.43MB | 3,862행
[7/225] 2008-07 | 기존 파일 정상 → 건너뛰기 | 6.43MB | 3,861행
[8/225] 2008-08 | 기존 파일 정상 → 건너뛰기 | 6.41MB | 3,852행
[9/225] 2008-09 | 기존 파일 정상 → 건너뛰기 | 6.38MB | 3,830행
[10/225] 2008-10 | 기존 파일 정상 → 건너뛰기 | 6.38MB | 3,829행
[11/225] 2008-11 | 기존 파일 정상 → 건너뛰기 | 6.38MB | 3,828행
[12/225] 2008-12 | 기존 파일 정상 → 건너뛰기 | 6.37MB | 3,826행
[13/225] 2009-01 | 기존 파일 정상 → 건너뛰기 | 6.36MB | 3,814행
[14/225] 2009-02 | 기존 파일 정상 → 건너뛰기 | 6.35MB | 3,812행
[15/225] 2009-03 | 기존 파일 정상 → 건너뛰기 | 6.35MB | 3,808행
[16/225] 2009-04 | 기존 파일 정상 → 건너뛰기 | 6.35MB | 3,808행
[17/225] 2009-05 | 기존 파일 정상 → 건너뛰기 | 6.34MB | 3,800행
[18/225] 2009-06 | 기존 파일 정상 → 건너뛰기 | 6.34MB | 3,801행
[19/225] 2009-07 | 기존 파일 정상 → 건너뛰기 | 6.34MB | 3,802행
[2

In [104]:
# 데이터가 있는 폴더로 작업 환경 변경
path = os.chdir('/Users/bestw/Desktop/ex/python_ex/PythonProject/개인학습/git_local_folder/공공데이터/인구분석/인구데이터')
files = [file for file in os.listdir(path) if file.endswith('월간.csv')]

files = sorted(
    file for file in os.listdir(path)
    if file.endswith('월간.csv')
)

df_result = []

except_col_list = ['_계_', '_총인구수', '_연령구간인구수']

for file in files:
    print(f'현재 읽는 파일은: {file}')

    # 모든 컬럼을 문자열로 읽어 숫자 파싱 오류 방지
    df = pl.read_csv(
        file,
        encoding='cp949',
        infer_schema=False
    ).lazy()

    df_cols = df.collect_schema().names()

    df_filter_cols = [
        col for col in df_cols
        if not any(word in col for word in except_col_list)
    ]

    df = (
        df.select(df_filter_cols)
        .with_columns(
            pl.col('행정구역')
            .str.split('(').list.first()
            .str.strip_chars()
            .str.split(' ')
            .alias('_지역목록')
        )
        .filter(
            pl.col('_지역목록')
            .list.last()
            .str.contains(r'[읍면동]$')
        )
        .unpivot(
            index=['행정구역', '_지역목록'],
            on=[col for col in df_filter_cols if col != '행정구역'],
            variable_name='연령대',
            value_name='인구수'
        )
        .with_columns(
            # "1,120" → "1120" → 정수 1120
            pl.col('인구수')
            .str.replace_all(',', '')
            .str.strip_chars()
            .cast(pl.Int64)
            .alias('인구수'),

            pl.col('행정구역')
            .str.extract(r'\((\d+)\)', 1)
            .alias('행정구역코드'),

            (
                pl.col('연령대').str.split('_').list.get(0)
                + pl.lit('01일')
            )
            .str.strptime(pl.Date, format='%Y년%m월%d일')
            .alias('날짜'),

            pl.col('연령대')
            .str.split('_').list.get(1)
            .alias('성별'),

            pl.col('연령대')
            .str.split('_').list.get(2)
            .str.extract(r'(\d+)', 1)
            .cast(pl.Int64)
            .alias('나이'),

            pl.col('_지역목록')
            .list.first()
            .alias('도/시'),

            pl.col('_지역목록')
            .list.slice(1, pl.col('_지역목록').list.len() - 2)
            .list.join(' ')
            .alias('시/구'),

            pl.col('_지역목록')
            .list.last()
            .alias('동')
        )
        .select([
            '날짜',
            '도/시',
            '시/구',
            '동',
            '행정구역코드',
            '성별',
            '나이',
            '인구수'
        ])
    )

    df_result.append(df)

    print('전처리 실행 계획 추가 완료')


# 실행 계획 합치기: collect() 없이 LazyFrame 유지
df = pl.concat(df_result, how='vertical_relaxed')

print(f'\n합친 파일 수: {len(df_result):,}')

현재 읽는 파일은: 200801_200801_연령별인구현황_월간.csv
전처리 실행 계획 추가 완료
현재 읽는 파일은: 200802_200802_연령별인구현황_월간.csv
전처리 실행 계획 추가 완료
현재 읽는 파일은: 200803_200803_연령별인구현황_월간.csv
전처리 실행 계획 추가 완료
현재 읽는 파일은: 200804_200804_연령별인구현황_월간.csv
전처리 실행 계획 추가 완료
현재 읽는 파일은: 200805_200805_연령별인구현황_월간.csv
전처리 실행 계획 추가 완료
현재 읽는 파일은: 200806_200806_연령별인구현황_월간.csv
전처리 실행 계획 추가 완료
현재 읽는 파일은: 200807_200807_연령별인구현황_월간.csv
전처리 실행 계획 추가 완료
현재 읽는 파일은: 200808_200808_연령별인구현황_월간.csv
전처리 실행 계획 추가 완료
현재 읽는 파일은: 200809_200809_연령별인구현황_월간.csv
전처리 실행 계획 추가 완료
현재 읽는 파일은: 200810_200810_연령별인구현황_월간.csv
전처리 실행 계획 추가 완료
현재 읽는 파일은: 200811_200811_연령별인구현황_월간.csv
전처리 실행 계획 추가 완료
현재 읽는 파일은: 200812_200812_연령별인구현황_월간.csv
전처리 실행 계획 추가 완료
현재 읽는 파일은: 200901_200901_연령별인구현황_월간.csv
전처리 실행 계획 추가 완료
현재 읽는 파일은: 200902_200902_연령별인구현황_월간.csv
전처리 실행 계획 추가 완료
현재 읽는 파일은: 200903_200903_연령별인구현황_월간.csv
전처리 실행 계획 추가 완료
현재 읽는 파일은: 200904_200904_연령별인구현황_월간.csv
전처리 실행 계획 추가 완료
현재 읽는 파일은: 200905_200905_연령별인구현황_월간.csv
전처리 실행 계획 추가 완료
현재 읽는 파일은: 200906_200906_연령별인구현황_월간.csv
전처리 실행 계

In [105]:
df = df.collect()